In [14]:
from langchain.tools import tool
from typing import List, Dict, Annotated
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor
from langchain_teddynote.messages import AgentStreamParser

import cv2
from ultralytics import YOLO

In [15]:
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()

True

In [17]:
yolo_model = YOLO("./checkpoints/yolo/yolo11n.pt")

# 도구 생성
@tool
def yolo_tool() -> List[Dict[str, str]]:
    """
    Object Detection 모델인 YOLO를 이용하여 현재 화면에서 정보 추출
    당신이 전방 카메라를 통해 시각 정보를 보고 싶다면 해당 도구를 사용하세요.
    """
    
    cap = cv2.VideoCapture(0)
    ret, frame = cap.read()
    frame = cv2.flip(frame, 1)
    results = yolo_model(source=frame, conf=0.5, verbose=False)

    info = {}
    for result in results:
        for box in result.boxes:
            cls = int(box.cls[0].item())
            label = yolo_model.names[cls]
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            x, y, w, h = map(int, box.xywh[0])
            confidence = round(box.conf[0].item(),2)
            info[label] = {
                'location': [x, y],
                'size': w * h,
                'bbox': [x1, y1, x2, y2],
                'confidence': confidence
            }
    return info

In [18]:
print(f"도구 이름: {yolo_tool.name}")
print(f"도구 설명: {yolo_tool.description}")

도구 이름: yolo_tool
도구 설명: Object Detection 모델인 YOLO를 이용하여 현재 화면에서 정보 추출
당신이 전방 카메라를 통해 시각 정보를 보고 싶다면 해당 도구를 사용하세요.


In [19]:
# tools 정의
tools = [yolo_tool]

In [6]:
# 프롬프트 생성
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. "
            "Make sure to use the `yolo_tool` tool for watching current view",
        ),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

In [7]:
# LLM 정의
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Agent 생성
agent = create_tool_calling_agent(llm, tools, prompt)

In [25]:
# AgentExecutor 생성
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=10,
    max_execution_time=10,
)

In [26]:
# AgentExecutor 실행
result = agent_executor.invoke({"input": "현재 무엇이 보이나요?"})

print("Agent 실행 결과:")
print(result["output"])



> Entering new AgentExecutor chain...

Invoking: `yolo_tool` with `{}`


{'person': {'location': [271, 295], 'size': 133955, 'bbox': [88, 112, 454, 479], 'confidence': 0.91}}현재 화면에는 한 명의 사람이 보입니다. 그 사람의 위치는 [271, 295]이며, 신뢰도는 91%입니다.

> Finished chain.
Agent 실행 결과:
현재 화면에는 한 명의 사람이 보입니다. 그 사람의 위치는 [271, 295]이며, 신뢰도는 91%입니다.


In [27]:
# session_id 를 저장할 딕셔너리 생성
store = {}

# session_id 를 기반으로 세션 기록을 가져오는 함수
def get_session_history(session_ids):
    if session_ids not in store:  # session_id 가 store에 없는 경우
        # 새로운 ChatMessageHistory 객체를 생성하여 store에 저장
        store[session_ids] = ChatMessageHistory()
    return store[session_ids]  # 해당 세션 ID에 대한 세션 기록 반환


# 채팅 메시지 기록이 추가된 에이전트를 생성합니다.
agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,
    # 대화 session_id
    get_session_history,
    # 프롬프트의 질문이 입력되는 key: "input"
    input_messages_key="input",
    # 프롬프트의 메시지가 입력되는 key: "chat_history"
    history_messages_key="chat_history",
)

agent_stream_parser = AgentStreamParser()

In [28]:
# 질의에 대한 답변을 스트리밍으로 출력 요청
response = agent_with_chat_history.stream(
    {"input": "안녕? 내 이름은 김현우야!"},
    # session_id 설정
    config={"configurable": {"session_id": "abc123"}},
)

# 출력 확인
for step in response:
    agent_stream_parser.process_agent_steps(step)



> Entering new None chain...
안녕하세요, 김현우님! 어떻게 도와드릴까요?

> Finished chain.
[최종 답변]
안녕하세요, 김현우님! 어떻게 도와드릴까요?


In [29]:
# 질의에 대한 답변을 스트리밍으로 출력 요청
response = agent_with_chat_history.stream(
    {"input": "현재 무엇이 보여?"},
    # session_id 설정
    config={"configurable": {"session_id": "abc123"}},
)

# 출력 확인
for step in response:
    agent_stream_parser.process_agent_steps(step)



> Entering new None chain...
[도구 호출]
Tool: yolo_tool
Log: 
Invoking: `yolo_tool` with `{}`




Invoking: `yolo_tool` with `{}`


{'person': {'location': [286, 301], 'size': 128502, 'bbox': [104, 124, 468, 479], 'confidence': 0.92}}[관찰 내용]
Observation: {'person': {'location': [286, 301], 'size': 128502, 'bbox': [104, 124, 468, 479], 'confidence': 0.92}}
현재 화면에는 한 사람이 보입니다. 그 사람의 위치는 [286, 301]이고, 신뢰도는 92%입니다. 추가로 궁금한 점이 있으신가요?

> Finished chain.
[최종 답변]
현재 화면에는 한 사람이 보입니다. 그 사람의 위치는 [286, 301]이고, 신뢰도는 92%입니다. 추가로 궁금한 점이 있으신가요?


In [30]:
# 질의에 대한 답변을 스트리밍으로 출력 요청
response = agent_with_chat_history.stream(
    {"input": "내 이름이 뭐라고?"},
    # session_id 설정
    config={"configurable": {"session_id": "abc123"}},
)

# 출력 확인
for step in response:
    agent_stream_parser.process_agent_steps(step)



> Entering new None chain...
당신의 이름은 김현우입니다!

> Finished chain.
[최종 답변]
당신의 이름은 김현우입니다!
